In [1]:
import os
import re
import sys
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from ase.io import read
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

# ---------- PORTABLE PATH SETTINGS ----------
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    # Jupyter Notebook fallback
    SCRIPT_DIR = Path.cwd()

# Resolves automatically relative to project structure
PROJECT_DIR = SCRIPT_DIR if (SCRIPT_DIR / "results").exists() else SCRIPT_DIR.parent
BASE_DIR = PROJECT_DIR / "data"
OUT_DIR  = PROJECT_DIR / "results"

FOLDER_NAMES    = ["r0", "r20", "r40", "r60", "r80", "r100", "r120", "r140", "r160", "r180",
                   "r200", "r220", "r240", "r260", "r280", "r300", "r320", "r340"]
SUBFOLDER_NAMES = ["t0", "t1.2", "t2.4", "t3.6", "t4.8", "t6.0", "t7.2", "t8.4", "t9.6"]
# --------------------------------------------


def is_jupyter():
    try:
        shell = get_ipython().__class__.__name__
        return shell == 'ZMQInteractiveShell'
    except NameError:
        return False


def validate_structure(atoms):
    if len(atoms) == 0:
        raise ValueError("Structure has zero atoms.")
    if atoms.get_cell().volume <= 0:
        raise ValueError("Invalid or zero-volume cell.")
    if np.isnan(atoms.get_positions()).any():
        raise ValueError("NaN detected in atomic positions.")


def wrap_positions_custom(positions, cell, pbc, center=(0.5, 0.5, 0.5)):
    """
    Consistent coordinate wrapping centered at 0.5 to keep layers 
    from splitting across boundary boundaries during clustering.
    """
    inv_cell = np.linalg.inv(cell)
    frac = np.dot(positions, inv_cell)
    for i in range(3):
        if pbc[i]:
            shift = frac[:, i] - center[i] + 0.5
            frac[:, i] = (shift % 1.0) + center[i] - 0.5
    return frac


def min_image_delta_frac(delta_frac, pbc):
    d = np.array(delta_frac, dtype=float, copy=True)
    for ax in range(3):
        if pbc[ax]:
            d[ax] -= np.round(d[ax])
    return d


def kmeans_1d_two_clusters(z, iters=15):
    z = np.asarray(z, dtype=float)
    if z.size < 2:
        raise ValueError("Not enough points for 2-cluster split")

    c0, c1 = float(z.min()), float(z.max())
    if np.isclose(c0, c1):
        raise ValueError("Degenerate z distribution")

    for _ in range(iters):
        d0 = np.abs(z - c0)
        d1 = np.abs(z - c1)
        m0 = d0 <= d1
        m1 = ~m0
        if m0.any():
            c0 = float(z[m0].mean())
        if m1.any():
            c1 = float(z[m1].mean())

    m0 = np.abs(z - c0) <= np.abs(z - c1)
    m1 = ~m0
    if not m0.any() or not m1.any():
        med = np.median(z)
        m0 = z <= med
        m1 = ~m0
        if not m0.any() or not m1.any():
            raise ValueError("Failed to split into 2 non-empty clusters")

    return m0, m1


def layer_centroids_frac(carbon_frac_wrapped):
    z = carbon_frac_wrapped[:, 2]
    m0, m1 = kmeans_1d_two_clusters(z)

    c0 = carbon_frac_wrapped[m0].mean(axis=0)
    c1 = carbon_frac_wrapped[m1].mean(axis=0)

    # Order Z coordinates: lower is always index 0, upper is index 1
    if c0[2] <= c1[2]:
        lower, upper = c0, c1
    else:
        lower, upper = c1, c0
    return lower, upper


def gather_cif_paths(base_dir, folders, subfolders):
    paths = []
    for folder in folders:
        for sub in subfolders:
            fp = os.path.join(base_dir, folder, sub)
            if not os.path.isdir(fp):
                continue
            for name in sorted(os.listdir(fp)):
                if name.lower().endswith(".cif"):
                    paths.append(os.path.join(fp, name))
    return paths


# ------------------- WORKER FUNCTION -------------------
def process_single_cif(cif_path):
    try:
        atoms = read(cif_path)
        validate_structure(atoms)

        symbols = np.array(atoms.get_chemical_symbols())
        positions = atoms.get_positions()
        pbc = np.array(atoms.get_pbc(), dtype=bool)
        cell = atoms.get_cell().array

        # Aligned wrapping (Centered on 0.5)
        frac_wrapped = wrap_positions_custom(positions, cell, pbc, center=(0.5, 0.5, 0.5))

        carbon_frac = frac_wrapped[symbols == "C"]
        if carbon_frac.shape[0] < 2:
            raise ValueError(f"Not enough carbon atoms ({carbon_frac.shape[0]})")

        centroid_lower_frac, centroid_upper_frac = layer_centroids_frac(carbon_frac)

        # Difference vector & minimum image convention
        delta_frac = centroid_upper_frac - centroid_lower_frac
        delta_frac = min_image_delta_frac(delta_frac, pbc)

        # Convert back to Cartesian angstroms
        TM_cart = delta_frac @ cell

        return {
            "status": "success",
            "file_path": cif_path,
            "X": float(TM_cart[0]),
            "Y": float(TM_cart[1]),
            "Z": float(TM_cart[2])
        }
    except Exception as e:
        return {
            "status": "error",
            "file_path": cif_path,
            "error": str(e)
        }


# ------------------- MAIN EXECUTION -------------------
def main():
    cif_paths = gather_cif_paths(BASE_DIR, FOLDER_NAMES, SUBFOLDER_NAMES)
    total_files = len(cif_paths)
    print(f"Found {total_files} CIF files to process.")

    # Determine optimal parallel executor
    if is_jupyter() and sys.platform.startswith("win"):
        Executor = ThreadPoolExecutor  # Prevents pickle errors in Windows Jupyter
        executor_type = "ThreadPoolExecutor"
    else:
        Executor = ProcessPoolExecutor
        executor_type = "ProcessPoolExecutor"

    num_workers = max(1, os.cpu_count() - 1)
    print(f"Processing using {num_workers} parallel workers ({executor_type})...")

    results = []
    errors = []

    with Executor(max_workers=num_workers) as executor:
        futures = {executor.submit(process_single_cif, p): p for p in cif_paths}
        
        with tqdm(total=total_files, desc="Calculating layer displacements", unit="file") as pbar:
            for future in as_completed(futures):
                res = future.result()
                if res["status"] == "success":
                    results.append({
                        "file_path": res["file_path"],
                        "X": res["X"],
                        "Y": res["Y"],
                        "Z": res["Z"]
                    })
                else:
                    errors.append((res["file_path"], res["error"]))
                pbar.update(1)

    # Save to outputs
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    df_results = pd.DataFrame(results)
    output_path = OUT_DIR / "cif_layer_displacements.csv"
    df_results.to_csv(output_path, index=False)

    print(f"\nCalculations complete!")
    print(f"Results saved to: {output_path}")
    print(df_results.head())

    if errors:
        print(f"\n--- Errors encountered during run ({len(errors)} total) ---")
        for path, msg in errors[:20]:
            print(f"- {path}: {msg}")


if __name__ == "__main__":
    main()


Found 2916 CIF files to process.
Processing using 7 parallel workers (ThreadPoolExecutor)...


Calculating layer displacements:   7%|▋         | 198/2916 [00:21<07:46,  5.82file/s]d:\New folder\project\.venv\Lib\site-packages\ase\io\cif.py:411: UserWarning: crystal system 'triclinic' is not interpreted for space group Spacegroup(1, setting=1). This may result in wrong setting!
  warnings.warn(
Calculating layer displacements: 100%|██████████| 2916/2916 [07:04<00:00,  6.87file/s]


Calculations complete!
Results saved to: d:\New folder\project\results\cif_layer_displacements.csv
                                     file_path         X         Y         Z
0    d:\New folder\project\data\r0\t0\t0_0.cif  0.116565 -1.063003  7.802620
1  d:\New folder\project\data\r0\t0\t0_160.cif  0.314323 -1.324229  8.924591
2  d:\New folder\project\data\r0\t0\t0_100.cif -0.322986 -1.384828  7.093224
3  d:\New folder\project\data\r0\t0\t0_180.cif  0.490655 -1.236403  9.484014
4   d:\New folder\project\data\r0\t0\t0_20.cif  0.111887 -0.880779  8.018344
